In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver = spark.table("workspace.default.silver_telemetry")

# 1. Average each robot's readings per hour (4.46M rows -> ~372K)
hourly = (silver
    .withColumn("hour", F.date_trunc("hour", "ts"))
    .groupBy("robot_id", "model", "hour")
    .agg(F.avg("battery_pct").alias("battery_pct"),
         F.avg("motor_temp_c").alias("motor_temp_c"),
         F.avg("loc_error_cm").alias("loc_error_cm")))

# 2. For each robot, look at its previous 24 hours
w = Window.partitionBy("robot_id").orderBy("hour").rowsBetween(-168, -1)

def zscore(col):
    return (F.col(col) - F.avg(col).over(w)) / F.stddev(col).over(w)

# 3. Flag hours far from that robot's normal
gold = (hourly
    .withColumn("z_temp", zscore("motor_temp_c"))
    .withColumn("z_loc", zscore("loc_error_cm"))
    .withColumn("z_batt", zscore("battery_pct"))
    .withColumn("is_anomaly", F.coalesce(
        (F.col("z_temp") > 3) | (F.col("z_loc") > 3) | (F.col("z_batt") < -3),
        F.lit(False))))

gold.write.mode("overwrite").saveAsTable("workspace.default.gold_robot_hourly")
print("gold rows:", gold.count())

gold rows: 372000


In [0]:
failures = spark.table("workspace.default.failure_log")
flagged = gold.filter("is_anomaly")

in_window = ((F.col("hour") >= F.date_trunc("hour", "start_ts")) &
             (F.col("hour") <= F.col("end_ts")))
matches = flagged.join(failures, "robot_id").filter(in_window)

recall = matches.select("failure_id").distinct().count() / failures.count()
precision = matches.select("robot_id", "hour").distinct().count() / flagged.count()

print(f"Recall:    {recall:.0%}  (share of the 40 failures we caught)")
print(f"Precision: {precision:.0%}  (share of our alerts that were real)")

Recall:    100%  (share of the 40 failures we caught)
Precision: 28%  (share of our alerts that were real)


In [0]:
# Precision 21%: about 4 out of 5 alerts were false alarms. In a real warehouse, operators would start ignoring the alerts within a week.

# Why so many false alarms: you’re running 3 tests on about 372,000 robot-hours. Even with a strict threshold of 3, random noise crosses it occasionally, and over that many hours, “occasionally” adds up to thousands of alerts. Real failures are a tiny fraction of the hours, so the random ones swamp them.

# The fix: require persistence. A random spike lasts one hour and disappears. A real failure keeps going. So only raise an alert when a robot is anomalous two hours in a row. That’s a common trick in real monitoring systems.

# The tradeoff: you’ll catch failures about an hour later, and you might miss a very short one. That’s exactly the precision vs. recall tradeoff interviewers like hearing you reason about.

In [0]:
# Tune

from pyspark.sql import functions as F
from pyspark.sql.window import Window

gold = spark.table("workspace.default.gold_robot_hourly")
failures = spark.table("workspace.default.failure_log")

# Alert only if this hour AND the previous hour were both anomalous
w = Window.partitionBy("robot_id").orderBy("hour")
gold = gold.withColumn(
    "is_alert",
    F.col("is_anomaly") & F.lag("is_anomaly", 1, False).over(w))

gold.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_robot_hourly")

# Re-score using alerts instead of single anomalies
flagged = gold.filter("is_alert")
in_window = ((F.col("hour") >= F.date_trunc("hour", "start_ts")) &
             (F.col("hour") <= F.col("end_ts")))
matches = flagged.join(failures, "robot_id").filter(in_window)

recall = matches.select("failure_id").distinct().count() / failures.count()
precision = matches.select("robot_id", "hour").distinct().count() / flagged.count()
print(f"Recall:    {recall:.0%}")
print(f"Precision: {precision:.0%}")

Recall:    92%
Precision: 69%


In [0]:
# Recall dropped because the new rule needs two anomalous hours in a row, and some failures can’t give you that:

# Short failures: a failure window can be as short as 2 hours. If only the second hour looks clearly abnormal, there’s no second anomalous hour to confirm it, so it never alerts.
# Slow starts: overheating ramps up gradually. Early hours are only slightly hot, so sometimes just the last hour crosses the threshold.

# So 4 failures you caught before now slip through. That’s the tradeoff you predicted: precision went from 21% to 56%, recall from 90% to 80%. You traded a few missed failures for far fewer false alarms.

In [0]:
# Second tune is this:
# In the first gold cell, change rowsBetween(-24, -1) to rowsBetween(-168, -1).
# Rerun that cell, then rerun the alert cell.